# Fused MoE operator benchmark

Compares the PyTorch expert-loop oracle with the Triton grouped-GEMM implementation. Results use the same per-GPU, per-operator timestamped archive as `operator_bench.ipynb`.

In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'tests':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(ROOT / 'tests'))

from minitrain.model.ops import get_ops_backend
from operator_bench_utils import (
    BenchCase, bench_sweep, correctness_stats, plot_kernel_grid, save_benchmark_results,
    to_summary_dataframe,
)

DEVICE = torch.device('cuda')
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
PROVIDERS = ('torch', 'triton')
WARMUP_MS, REP_MS = 25, 100
BENCHMARK_CACHE_DIR = ROOT / 'tests' / 'benchmark_results'
FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)


## Router postprocess correctness

The fused path must match the Torch oracle for Top-K weights, expert probability statistics, z-loss, entropy, and the combined gradient from all differentiable outputs. Unsupported devices or deterministic mode intentionally use the backend fallback.

In [ ]:
from minitrain.kernels.triton.router import is_router_postprocess_supported

torch.manual_seed(17)
TOKENS, ROUTER_EXPERTS, ROUTER_TOP_K = 257, 64, 2
base_logits = torch.randn(TOKENS, ROUTER_EXPERTS, device=DEVICE, dtype=torch.float32)
weight_grad = torch.randn(TOKENS, ROUTER_TOP_K, device=DEVICE)
probability_grad = torch.randn(ROUTER_EXPERTS, device=DEVICE)

def run_router_with_grad(provider):
    logits = base_logits.detach().clone().requires_grad_(True)
    route = get_ops_backend(provider).router_postprocess(
        logits, ROUTER_TOP_K, normalize=True
    )
    objective = (
        (route.expert_weights * weight_grad).sum()
        + (route.probability_per_expert * probability_grad).sum()
        + 0.13 * route.z_loss
    )
    objective.backward()
    return route, logits.grad

reference, reference_grad = run_router_with_grad('torch')
candidate, candidate_grad = run_router_with_grad('triton')
torch.testing.assert_close(candidate.expert_indices, reference.expert_indices)
torch.testing.assert_close(candidate.expert_weights, reference.expert_weights, atol=2e-5, rtol=2e-5)
torch.testing.assert_close(candidate.probability_per_expert, reference.probability_per_expert, atol=2e-5, rtol=2e-5)
torch.testing.assert_close(candidate.z_loss, reference.z_loss, atol=2e-5, rtol=2e-5)
torch.testing.assert_close(candidate.entropy, reference.entropy, atol=2e-5, rtol=2e-5)
torch.testing.assert_close(candidate_grad, reference_grad, atol=5e-5, rtol=5e-5)
previous_deterministic = torch.are_deterministic_algorithms_enabled()
try:
    torch.use_deterministic_algorithms(True)
    assert not is_router_postprocess_supported(base_logits, ROUTER_TOP_K, True)
    deterministic_route = get_ops_backend('triton').router_postprocess(
        base_logits, ROUTER_TOP_K, normalize=True
    )
    torch.testing.assert_close(
        deterministic_route.expert_weights, reference.expert_weights
    )
finally:
    torch.use_deterministic_algorithms(previous_deterministic)
print('native Triton path:', is_router_postprocess_supported(base_logits, ROUTER_TOP_K, True))
print('router forward and combined backward: OK')


## Dropless end-to-end MoE smoke

Capacity masking is disabled until dispatch compaction exists in every backend. This check runs a complete dropless MoE transformer backward through the selected backend, using BF16 when the GPU supports it and FP16 otherwise.

In [ ]:
from minitrain.model.config import ModelConfig
from minitrain.model.transformer import MiniTransformer

smoke_cfg = ModelConfig(
    vocab_size=128, seq_len=16, n_layers=2, n_heads=4, hidden_size=64,
    intermediate_size=128, ffn_type='moe', num_experts=8, experts_per_token=2,
)
model = MiniTransformer(smoke_cfg, get_ops_backend('triton'), activation_dtype=DTYPE).to(DEVICE)
input_ids = torch.randint(0, smoke_cfg.vocab_size, (2, smoke_cfg.seq_len), device=DEVICE)
with torch.autocast('cuda', dtype=DTYPE):
    loss, _ = model(input_ids, input_ids)
loss.backward()
router_grads = [block.ffn.router.weight.grad for block in model.blocks]
assert all(grad is not None and torch.isfinite(grad).all() for grad in router_grads)
assert smoke_cfg.expert_capacity_factor is None
print('dropless end-to-end MoE backward: OK')


## Router postprocess benchmark

In [ ]:
def make_router_case(tokens):
    return BenchCase(
        tensors={'logits': torch.randn(tokens, ROUTER_EXPERTS, device=DEVICE, dtype=torch.float32)},
        grad_names=('logits',),
    )

def router_forward(provider, tensors):
    route = get_ops_backend(provider).router_postprocess(
        tensors['logits'], ROUTER_TOP_K, normalize=True
    )
    return route.expert_weights, route.probability_per_expert, route.z_loss

router_rows = bench_sweep(
    kernel='router_postprocess', providers=PROVIDERS,
    sizes=(128, 512, 2048, 8192, 32768),
    size_label=f'tokens; E={ROUTER_EXPERTS}, K={ROUTER_TOP_K}',
    make_case=make_router_case, forward=router_forward,
    warmup_ms=WARMUP_MS, rep_ms=REP_MS, atol=5e-5, rtol=5e-5,
)
router_cache = save_benchmark_results(
    router_rows, benchmark='router_postprocess', cache_root=BENCHMARK_CACHE_DIR,
    metadata={'num_experts': ROUTER_EXPERTS, 'top_k': ROUTER_TOP_K},
)
print(f'raw benchmark data: {router_cache}')
display(to_summary_dataframe(router_rows))
_ = plot_kernel_grid(router_rows, save_path=FIG_DIR / 'router_postprocess_summary.png')


In [ ]:
NUM_EXPERTS, HIDDEN, INTERMEDIATE, TOP_K = 8, 256, 128, 2

def make_fused_moe_case(tokens):
    router_logits = torch.randn(tokens, NUM_EXPERTS, device=DEVICE)
    top_k_logits, top_k_index = torch.topk(router_logits, TOP_K, dim=-1)
    top_k_weights = torch.softmax(top_k_logits, dim=-1).to(DTYPE)
    return BenchCase(
        tensors={
            'x': torch.randn(tokens, HIDDEN, device=DEVICE, dtype=DTYPE),
            'gate_up_proj': torch.randn(NUM_EXPERTS, 2 * INTERMEDIATE, HIDDEN, device=DEVICE, dtype=DTYPE) * 0.02,
            'down_proj': torch.randn(NUM_EXPERTS, HIDDEN, INTERMEDIATE, device=DEVICE, dtype=DTYPE) * 0.02,
            'top_k_index': top_k_index.to(torch.int32),
            'top_k_weights': top_k_weights,
        },
        grad_names=('x', 'gate_up_proj', 'down_proj', 'top_k_weights'),
    )

def fused_moe_forward(provider, tensors):
    return get_ops_backend(provider).fused_moe(
        tensors['x'], tensors['gate_up_proj'], tensors['down_proj'],
        tensors['top_k_index'], tensors['top_k_weights'],
    )

def make_fused_moe_edge_case(tokens):
    edge_top_k = 3
    top_k_index = torch.tensor(
        [[(token + slot) % 2 for slot in range(edge_top_k)] for token in range(tokens)],
        device=DEVICE, dtype=torch.int32,
    )
    top_k_weights = torch.rand(tokens, edge_top_k, device=DEVICE, dtype=DTYPE)
    top_k_weights[::2, -1] = 0
    return BenchCase(
        tensors={
            'x': torch.randn(tokens, HIDDEN, device=DEVICE, dtype=DTYPE),
            'gate_up_proj': torch.randn(NUM_EXPERTS, 2 * INTERMEDIATE, HIDDEN, device=DEVICE, dtype=DTYPE) * 0.02,
            'down_proj': torch.randn(NUM_EXPERTS, HIDDEN, INTERMEDIATE, device=DEVICE, dtype=DTYPE) * 0.02,
            'top_k_index': top_k_index,
            'top_k_weights': top_k_weights,
        },
        grad_names=('x', 'gate_up_proj', 'down_proj', 'top_k_weights'),
    )

edge_stats = correctness_stats(
    make_fused_moe_edge_case, 33, 'triton', fused_moe_forward, atol=3e-2, rtol=8e-2,
)
assert edge_stats['fwd_correct'] and edge_stats['bwd_correct'], edge_stats
print('fused MoE skewed/empty-expert, K=3, zero-weight gradient probe: OK')


In [ ]:
fused_moe_rows = bench_sweep(
    kernel='fused_moe',
    providers=PROVIDERS,
    sizes=(128, 256, 512, 1024, 2048),
    size_label=f'tokens; E={NUM_EXPERTS}, H={HIDDEN}, I={INTERMEDIATE}, K={TOP_K}',
    make_case=make_fused_moe_case,
    forward=fused_moe_forward,
    warmup_ms=WARMUP_MS,
    rep_ms=REP_MS,
    atol=5e-2,
    rtol=5e-2,
)
cache_path = save_benchmark_results(
    fused_moe_rows, benchmark='fused_moe', cache_root=BENCHMARK_CACHE_DIR,
    metadata={'dtype': str(DTYPE), 'num_experts': NUM_EXPERTS, 'hidden': HIDDEN, 'intermediate': INTERMEDIATE, 'top_k': TOP_K},
)
print(f'raw benchmark data: {cache_path}')
display(to_summary_dataframe(fused_moe_rows))
_ = plot_kernel_grid(fused_moe_rows, save_path=FIG_DIR / 'fused_moe_summary.png')
